In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina_27_Dates_Measurement_Catalogue.pdf
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina_27_Dates_Manual_English_2026.pdf
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina_27_Dates_Manual_Arabic_2026.pdf
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/License-v2.txt
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina 27 Dates Hackathon Dataset v1 in .PNG -Catagorized/Balah_HalwahS24/s24.o24.9.png
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina 27 Dates Hackathon Dataset v1 in .PNG -Catagorized/Balah_HalwahS24/s24.r2.png
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina 27 Dates Hackathon Dataset v1 in .PNG -Catagorized/Balah_HalwahS24/s24.o24.5.png
/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1/Medina 27 D

In [2]:
   import os, shutil

   BASE = "/kaggle/input/datasets/firasaalsaidi/medina-27-dates-hackathon-dataset-v1"
   DST = "/kaggle/working/images_dataset"

   for split in ["train", "validation", "test"]:
       for cls in sorted(os.listdir(os.path.join(BASE, split))):
           src_cls = os.path.join(BASE, split, cls)
           if os.path.isdir(src_cls):
               os.makedirs(os.path.join(DST, cls), exist_ok=True)
               for f in os.listdir(src_cls):
                   shutil.copy(os.path.join(src_cls, f), os.path.join(DST, cls, f"{split}_{f}"))
   print(sum(len(fs) for _, _, fs in os.walk(DST)), "images")

1161 images


In [3]:
%%writefile sequential_process.py
import os
import cv2
import time
import numpy as np
from PIL import Image, ImageDraw, ImageFont

cv2.setNumThreads(1)   # ADDED: force OpenCV to use one thread, so the baseline is truly single-core

# CONFIGURATION
INPUT_DIR = "/kaggle/working/images_dataset"                      # Input dataset folder (class folders inside)
OUTPUT_DIR = "sample_output/output_seq"           # Output folder: resized + watermarked images
EDGES_DIR = "sample_output/output_seq_edges"      # Output folder: Sobel edge images (ADDED)
WATERMARK_TEXT = "ARTI 503 Parallel Project"      # Watermark text (CHANGED)
RESIZE_SHAPE = (128, 128)                         # Resize dimensions (width, height)
FONT_PATH = None
BLUR_KERNEL = (5, 5)                              # Gaussian blur window size (ADDED)


# MAIN PROCESS
def add_watermark(image_pil, text, font_path=None):
    """Add a simple white watermark at the bottom-right corner."""
    draw = ImageDraw.Draw(image_pil)

    if font_path and os.path.exists(font_path):
        font = ImageFont.truetype(font_path, 14)
    else:
        font = ImageFont.load_default()

    # Compatible with both old and new Pillow versions
    try:
        # New Pillow versions (>=10)
        left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
        text_w, text_h = right - left, bottom - top
    except AttributeError:
        # Older Pillow versions
        text_w, text_h = draw.textsize(text, font=font)

    x, y = image_pil.width - text_w - 8, image_pil.height - text_h - 8
    draw.text((x, y), text, fill=(255, 255, 255), font=font)
    return image_pil


def detect_edges(img_bgr):
    """ADDED: grayscale -> Gaussian blur -> Sobel edge detection.
    Returns an 8-bit edge image."""
    # 1. Grayscale conversion
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian blur
    blurred = cv2.GaussianBlur(gray, BLUR_KERNEL, 0)

    # 3. Sobel edge detection (horizontal + vertical, then combine)
    grad_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(grad_x, grad_y)

    # Scale to 0-255 so it can be saved as a normal image
    edges = cv2.convertScaleAbs(magnitude)
    return edges


def sequential_preprocess():
    """Perform sequential image processing:
    grayscale, blur, Sobel (added), then resizing and watermarking (original)."""
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    if not os.path.exists(EDGES_DIR):          # ADDED
        os.makedirs(EDGES_DIR)

    start_time = time.time()

    # Loop through all class folders
    for class_name in os.listdir(INPUT_DIR):
        class_path = os.path.join(INPUT_DIR, class_name)
        if not os.path.isdir(class_path):
            continue

        # Create corresponding output folders
        output_class_path = os.path.join(OUTPUT_DIR, class_name)
        os.makedirs(output_class_path, exist_ok=True)
        edges_class_path = os.path.join(EDGES_DIR, class_name)   # ADDED
        os.makedirs(edges_class_path, exist_ok=True)             # ADDED

        # Loop through all images
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Skipping invalid image: {img_path}")
                    continue

                # ADDED: grayscale + blur + Sobel on the original image
                edges = detect_edges(img)
                cv2.imwrite(os.path.join(edges_class_path, img_name), edges)

                # Resize
                img_resized = cv2.resize(img, RESIZE_SHAPE)

                # Convert to PIL (for watermark)
                img_pil = Image.fromarray(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
                img_pil = add_watermark(img_pil, WATERMARK_TEXT, FONT_PATH)

                # Convert back to OpenCV format
                final_img = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

                # Save processed image
                output_path = os.path.join(output_class_path, img_name)
                cv2.imwrite(output_path, final_img)

            except Exception as e:
                print(f"Error processing {img_path}: {e}")

    end_time = time.time()
    print(f"Sequential Processing Time: {end_time - start_time:.2f} seconds")


if __name__ == "__main__":
    sequential_preprocess()

Writing sequential_process.py


In [4]:
!python sequential_process.py

Sequential Processing Time: 27.06 seconds


In [5]:
   import os
   for d in ["sample_output/output_seq", "sample_output/output_seq_edges"]:
       print(d, sum(len(f) for _, _, f in os.walk(d)))

sample_output/output_seq 1161
sample_output/output_seq_edges 1161
